# Laboratory Day 9b: Semantic Segmentation (U-Net): Modeling, Training

In this exercise, we will implement and train a U-Net using Tensorlow. In the last exercise, we have prepared tfrecords, which should be used in this exercise.

For deep learning task, there is usually no unique solution. Feel free when you write your own implementation.

To solve the task here, all functions in Tensorflow Lib are allowed be used/called.


## 1. Modeling and Loss Function
**Exercise 1.1 (10 points)**
1. Build the U-Net structure from scratch and print out model.summary().
2. Implement the cross-entropy loss function.

In [1]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Input, UpSampling2D, concatenate, BatchNormalization, Activation
from tensorflow.keras.models import Model
import tensorflow as tf

def conv_block(input_tensor, num_filters):
    x = Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(num_filters, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

def modeling(img_h, img_w, img_c, num_class):
    x_input = Input(shape=(img_h, img_w, img_c))

    # Encoder (Contracting Path)
    conv1 = conv_block(x_input, 64)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = conv_block(pool1, 128)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = conv_block(pool2, 256)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = conv_block(pool3, 512)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)

    # Bottleneck
    conv5 = conv_block(pool4, 1024)

    # Decoder (Expansive Path)
    up6 = concatenate([UpSampling2D(size=(2, 2))(conv5), conv4], axis=-1)
    conv6 = conv_block(up6, 512)

    up7 = concatenate([UpSampling2D(size=(2, 2))(conv6), conv3], axis=-1)
    conv7 = conv_block(up7, 256)

    up8 = concatenate([UpSampling2D(size=(2, 2))(conv7), conv2], axis=-1)
    conv8 = conv_block(up8, 128)

    up9 = concatenate([UpSampling2D(size=(2, 2))(conv8), conv1], axis=-1)
    conv9 = conv_block(up9, 64)

    # Output layer
    y_out = Conv2D(num_class, (1, 1), activation='softmax')(conv9) # Using softmax for multi-class segmentation

    # Modeling
    model = Model(inputs=x_input, outputs=y_out)
    model.summary()

    return model

In [2]:
import tensorflow as tf

def cross_entropy_loss(image_mask, y_pred):

    if image_mask.shape[-1] == 1:
        image_mask = tf.squeeze(image_mask, axis=-1)

    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

    # Calculate the loss
    loss = loss_fn(image_mask, y_pred)


    return loss

## 2. Training
**Exercise 2.1 (20 points)**:
1. Write a training program to train the model for in total $N$ steps. Choose the proper learning rate such that the loss function can be effectively reduced.

2. For each $S$ training steps (e.g. $S$=1000),

    - Print out (a) the average loss of the training dataset for the last $S$ steps.
    
    - Optional: Compute the confusion matrix (based on the classification results of each pixel)
       

3. After each $S$ training steps, evaluate the average loss of the validation data set.

    - Print out (a) the average loss of the entire validation dataset.
    
    - Optional: Compute the confusion matrix. Compute precision, recall as metrics.
    
   
4. Please print out all debugging info and metrics evaluation results into a *.txt file. It is recommended to use logging package for recording longer training process (optional, see below).

5. Optional task: One can also visualize the training process in tensorboard.

    - Tensorboard can be started by running this command:  tensorboard --logdir="path_to_dir" --bind_all
    
    - Using tensorboard is recommended, but not mandatory.
    

6. Implement save_model() and load_model() to save and load the model.


U-Net is not a trivial system, and training it is complicated. Any mistakes in the data pipeline, in the modeling and in the selection of training parameters may lead to undesired training results. To debug the training process, my suggestions:

1. Visualize the images and the image masks, which are directly feed into network.

2. Visualize the output of the network. If the network learns, the predicted mask should converge to the GT mask.

3. Obverse the loss function of training set and validation set, which should get minimized by the optimizer.

4. Be careful of overfitting. Use data augmentation, batch normalization, regularization, dropout layers, early stop to fight against overfitting.

4. Check the implementation of the cross-entropy loss function.

5. Adjust the learning rate. Small learning rates may result in a more stable training process.

Hope you could get a working U-Net.

In [ ]:
# functions, which could be used
import logging
def init_logging(log_path=None, mode='w', level=logging.INFO):
    if not log_path:
        log_path = os.path.join(os.getcwd(), 'filename.log')
    if level is None:
        level = logging.INFO
    format = '%(asctime)s - %(name)s - %(levelname)s : %(message)s'
    handlers = [logging.FileHandler(log_path, mode=mode), logging.StreamHandler()]
    logging.basicConfig(level=level, format=format, handlers=handlers)
    logging.info(f'log_path: {log_path}')


In [ ]:


def init_logging(log_path=None, mode='w', level=logging.INFO):
    if not log_path:
        log_path = os.path.join(os.getcwd(), 'filename.log')
    if level is None:
        level = logging.INFO
    format = '%(asctime)s - %(name)s - %(levelname)s : %(message)s'
    handlers = [logging.FileHandler(log_path, mode=mode), logging.StreamHandler()]
    logging.basicConfig(level=level, format=format, handlers=handlers)
    logging.info(f'log_path: {log_path}')

# Moved from cell 2b659514 (U-Net architecture definitions)
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Input, UpSampling2D, concatenate, BatchNormalization, Activation
from tensorflow.keras.models import Model

def conv_block(input_tensor, num_filters):
    x = Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(num_filters, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

def modeling(img_h, img_w, img_c, num_class):
    # print("Defining U-Net model...") # Debug print
    x_input = Input(shape=(img_h, img_w, img_c))

    # Encoder (Contracting Path)
    conv1 = conv_block(x_input, 64)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = conv_block(pool1, 128)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = conv_block(pool2, 256)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = conv_block(pool3, 512)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)

    # Bottleneck
    conv5 = conv_block(pool4, 1024)

    # Decoder (Expansive Path)
    up6 = concatenate([UpSampling2D(size=(2, 2))(conv5), conv4], axis=-1)
    conv6 = conv_block(up6, 512)

    up7 = concatenate([UpSampling2D(size=(2, 2))(conv6), conv3], axis=-1)
    conv7 = conv_block(up7, 256)

    up8 = concatenate([UpSampling2D(size=(2, 2))(conv7), conv2], axis=-1)
    conv8 = conv_block(up8, 128)

    up9 = concatenate([UpSampling2D(size=(2, 2))(conv8), conv1], axis=-1)
    conv9 = conv_block(up9, 64)

    # Output layer
    y_out = Conv2D(num_class, (1, 1), activation='softmax')(conv9) # Using softmax for multi-class segmentation

    # Modeling
    model = Model(inputs=x_input, outputs=y_out)
    # model.summary() # Removed summary to reduce output clutter during execution, can be re-enabled for inspection

    return model

# Moved from cell 2e81ade8 (Cross-entropy loss function)
def cross_entropy_loss(image_mask, y_pred):
    if image_mask.shape[-1] == 1:
        image_mask = tf.squeeze(image_mask, axis=-1)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
    loss = loss_fn(image_mask, y_pred)
    return loss


work_dir = './unet_model_training' # Set a default working directory

# Ensure the work directory exists
if not os.path.exists(work_dir):
    os.makedirs(work_dir)

# logging setup
logging_path = os.path.join(work_dir, 'logging.log')
init_logging(logging_path, level=logging.DEBUG)

logger = logging.getLogger(__name__)


class UnetModel(tf.keras.Model):
    def __init__(self, work_dir):
        super().__init__()

        self.input_w = 512
        self.input_h = 512
        self.input_c = 3
        self.num_class = 2

        self._setup_layers()
        self._setup_metrics()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.5)
        self.work_dir = work_dir

    @property
    def metrics(self):
        return [self._loss_metric]

    def _setup_metrics(self):
        self._loss_metric = tf.keras.metrics.Mean(name='loss_mean')

    def _metrics_reset(self):
        for m in self.metrics:
            m.reset_states()

    def _update_metrics(self, loss):
        self._loss_metric.update_state(loss)

    def _setup_layers(self):

        self.cnn = modeling(self.input_h, self.input_w, self.input_c, self.num_class)

    def train(self, data_train, steps, delta_steps=100):
        logger.info(f'Starting training for {steps} steps.')
        for step, (image, y_label) in enumerate(data_train):
            if step >= steps:
                logger.info(f'Reached maximum steps: {steps}')
                break

            with tf.GradientTape() as tape:
                y_pred = self.cnn(image, training=True) # Forward pass
                loss = self.loss_fun(y_pred, y_label) # compute loss

            trainable_vars = self.trainable_variables
            gradients = tape.gradient(loss, trainable_vars)
            self.optimizer.apply_gradients(zip(gradients, trainable_vars))

            # update metrics
            self._update_metrics(loss)

            if (step + 1) % delta_steps == 0:
                avg_loss = self._loss_metric.result().numpy()
                logger.info(f'Step {step+1}/{steps}: Training Loss = {avg_loss:.4f}')
                self._metrics_reset()

        logger.info(f'Done training: total_step {step+1}, finish')
        # Save model after training
        self.save_model(model_name='final_unet_model.keras') # Changed to .keras


    def loss_fun(self, y_pred, y_label):
        return cross_entropy_loss(y_label, y_pred)

    def data_visualize(self, data):
        raise NotImplementedError # Not implemented yet

    def save_model(self, model_name='unet_model.keras'): # Changed default to .keras
        model_save_path = os.path.join(self.work_dir, model_name)
        logger.info(f"Attempting to save model to: {model_save_path}") # Debug log
        self.cnn.save(model_save_path)
        if os.path.exists(model_save_path): # Verify file creation
            logger.info(f"Model successfully saved to {model_save_path}")
        else:
            logger.error(f"Failed to save model to {model_save_path}. File does not exist after save call.")

    def load_model(self, model_path):
        self.cnn = tf.keras.models.load_model(model_path, compile=False) # Added compile=False
        logger.info(f"Model loaded from {model_path}")




# Instantiate the model with the defined work_dir
model = UnetModel(work_dir=work_dir)



# Call the train method
model.train(data_train=train_dataset, steps=10, delta_steps=5)

## 3. Prediction
**Exercise 3.1 (5 points)**:
1. Load the U-Net model by the best Checkpoint or SavedModel, which are obtained after training.
2. Do prediction on 2-3 new images, which have not seen by the model. Visualize the image mask and prediction.

In [1]:


def conv_block(input_tensor, num_filters):
    x = Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(num_filters, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

def modeling(img_h, img_w, img_c, num_class):
    x_input = Input(shape=(img_h, img_w, img_c))

    # Encoder (Contracting Path)
    conv1 = conv_block(x_input, 64)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = conv_block(pool1, 128)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = conv_block(pool2, 256)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = conv_block(pool3, 512)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)

    # Bottleneck
    conv5 = conv_block(pool4, 1024)

    # Decoder (Expansive Path)
    up6 = concatenate([UpSampling2D(size=(2, 2))(conv5), conv4], axis=-1)
    conv6 = conv_block(up6, 512)

    up7 = concatenate([UpSampling2D(size=(2, 2))(conv6), conv3], axis=-1)
    conv7 = conv_block(up7, 256)

    up8 = concatenate([UpSampling2D(size=(2, 2))(conv7), conv2], axis=-1)
    conv8 = conv_block(up8, 128)

    up9 = concatenate([UpSampling2D(size=(2, 2))(conv8), conv1], axis=-1)
    conv9 = conv_block(up9, 64)

    # Output layer
    y_out = Conv2D(num_class, (1, 1), activation='softmax')(conv9) # Using softmax for multi-class segmentation

    # Modeling
    model = Model(inputs=x_input, outputs=y_out)

    return model



# UnetModel class definition, moved from fe780e3f
class UnetModel(tf.keras.Model):
    def __init__(self, work_dir):
        super().__init__()
        self.input_w = 512
        self.input_h = 512
        self.input_c = 3
        self.num_class = 2

        self._setup_layers()
        self._setup_metrics()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.5)

        self.work_dir = work_dir

    @property
    def metrics(self):
        return [self._loss_metric]

    def _setup_metrics(self):
        self._loss_metric = tf.keras.metrics.Mean(name='loss_mean')

    def _metrics_reset(self):
        for m in self.metrics:
            m.reset_states()

    def _update_metrics(self, loss):
        self._loss_metric.update_state(loss)

    def _setup_layers(self):
        self.cnn = modeling(self.input_h, self.input_w, self.input_c, self.num_class)

    def train(self, data_train, steps, delta_steps=100):
        logger.info(f'Starting training for {steps} steps.')
        for step, (image, y_label) in enumerate(data_train):
            if step >= steps:
                logger.info(f'Reached maximum steps: {steps}')
                break

            with tf.GradientTape() as tape:
                y_pred = self.cnn(image, training=True)
                loss = self.loss_fun(y_pred, y_label)

            trainable_vars = self.trainable_variables
            gradients = tape.gradient(loss, trainable_vars)
            self.optimizer.apply_gradients(zip(gradients, trainable_vars))

            self._update_metrics(loss)

            if (step + 1) % delta_steps == 0:
                avg_loss = self._loss_metric.result().numpy()
                logger.info(f'Step {step+1}/{steps}: Training Loss = {avg_loss:.4f}')
                self._metrics_reset()

        logger.info(f'Done training: total_step {step+1}, finish')
        self.save_model(model_name='final_unet_model.keras') # Changed to .keras

    def loss_fun(self, y_pred, y_label):
        return cross_entropy_loss(y_label, y_pred)

    def data_visualize(self, data):
        raise NotImplementedError

    def save_model(self, model_name='unet_model.keras'): # Changed default to .keras
        model_save_path = os.path.join(self.work_dir, model_name)
        self.cnn.save(model_save_path)
        logger.info(f"Model saved to {model_save_path}")

    def load_model(self, model_path):
        self.cnn = tf.keras.models.load_model(model_path, compile=False) # Added compile=False
        logger.info(f"Model loaded from {model_path}")



work_dir = './unet_model_training'
model_name = 'final_unet_model.keras' # Changed to .keras
model_path = os.path.join(work_dir, model_name)

# Ensure the work directory exists, though it should have been created by the training cell
if not os.path.exists(work_dir):
    logger.warning(f"Work directory {work_dir} not found. Creating it.")
    os.makedirs(work_dir)

input_h = 512
input_w = 512
input_c = 3
num_class = 2 # Assuming binary segmentation

# Generate a simple image with a distinct object
new_image_raw = np.zeros((input_h, input_w, input_c), dtype=np.float32)
# Draw a white circle
center_x, center_y = input_w // 2, input_h // 2
radius = 100
Y, X = np.ogrid[:input_h, :input_w]
dist_from_center = np.sqrt((X - center_x)**2 + (Y - center_y)**2)
mask = dist_from_center < radius
new_image_raw[mask, :] = 1.0 # Set circle to white

new_image_for_prediction = tf.constant(new_image_raw[np.newaxis, ...], dtype=tf.float32) # Add batch dimension


loaded_model_instance = UnetModel(work_dir=work_dir)

# Load the trained model
try:
    loaded_model_instance.load_model(model_path)
    logger.info("Model successfully loaded for prediction.")
except Exception as e:
    logger.error(f"Error loading model: {e}. Please ensure the model was saved correctly after training.")
    raise e # Re-raise for visibility if running in a notebook

# Perform prediction
# The `cnn` attribute holds the loaded Keras Model
prediction_output = loaded_model_instance.cnn.predict(new_image_for_prediction)

# Process the prediction:
# Assuming softmax output, get the class with highest probability for each pixel
predicted_mask = tf.argmax(prediction_output, axis=-1)
# Squeeze the batch dimension (and channel if it was (1, H, W, 1))
predicted_mask = tf.squeeze(predicted_mask, axis=0) # remove batch dimension
predicted_mask_np = predicted_mask.numpy().astype(np.float32) # Convert to numpy for visualization

# Visualize the original image and the predicted mask
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(new_image_raw) # Assuming the dummy image is in [0, 1] range
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(predicted_mask_np, cmap='gray') # Display mask in grayscale
plt.title('Predicted Mask')
plt.axis('off')

plt.show()

logger.info(f"Prediction complete. Visualizing original image and predicted mask.")

ERROR:__main__:Error loading model: File not found: filepath=./unet_model_training/final_unet_model.keras. Please ensure the file is an accessible `.keras` zip file.. Please ensure the model was saved correctly after training.


ValueError: File not found: filepath=./unet_model_training/final_unet_model.keras. Please ensure the file is an accessible `.keras` zip file.